# **Project 1| Global Baseline Predictors and RSME**

# by Sandra Dela Cruz

This system recommends a set of artist's oil paints to professional artists.

In [ ]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

#create toy dataset
ratings = pd.DataFrame({
    'Gamblin':         [5, 4, 2, np.nan, 3, 1],
    'Winsor & Newton': [np.nan, 3, 4, 5, 2, 2],
    'Utrecht':         [np.nan, 3, 3, 5, 1, np.nan],
    'Holbein':         [5, 5, np.nan, 2, 5, 4],
    'Sennelier':       [4, 2, 4, 4, 3, 3]

}, index = ['A', 'B', 'C', 'D', 'E', 'F'])

ratings

In [ ]:
# split into training and test dataset

# Create test dataset
test = pd.DataFrame([
    ('C', 'Gamblin', 2),
    ('B', 'Winsor & Newton', 3),
    ('E', 'Utrecht', 1),
    ('A', 'Holbein', 5),
    ('D', 'Sennelier', 4)
], columns=['user', 'item', 'rating'])

test

In [ ]:
# Remove test data information from original dataset
train = ratings.copy()

for user, item, rating in test.itertuples(index=False):
    train.loc[user, item] = np.nan

train

In [ ]:
# Calculate raw average (mean) rating on training data
mean_rating = train.stack().mean()

mean_rating

In [ ]:
# Build RSME function
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

# Get RMSE for raw average of train set
train_preds = []
train_true = []

for i in train.stack().index:
    train_true.append(train.loc[i])
    train_preds.append(mean_rating)

train_rmse = rmse(np.array(train_true), np.array(train_preds))

train_rmse.round(4)

In [ ]:
# Get RMSE for raw average of test set
test_rmse = rmse(
    test['rating'].values,
    np.full(len(test), mean_rating)
)

test_rmse.round(4)

In [ ]:
# Compute user and item biases
def compute_biases(train, mean_rating):
    user_bias = pd.Series(0.0, index=train.index)
    item_bias = pd.Series(0.0, index=train.columns)

    # update user bias
    for u in train.index:
        ratings = train.loc[u].dropna()
        if len(ratings) > 0:
            user_bias[u] = (ratings - mean_rating).mean()

    # update item bias
    for i in train.columns:
        ratings = train[i].dropna()
        if len(ratings) > 0:
            item_bias[i] = (ratings - mean_rating).mean()

    return user_bias, item_bias

user_bias, item_bias = compute_biases(train, mean_rating)

user_bias.round(2), item_bias.round(2)

In [ ]:
# Build function to calculate the baseline predictors
def baseline_predictor(user, item, mean_rating, user_bias, item_bias):
    pred = mean_rating + user_bias.get(user, 0) + item_bias.get(item, 0)
    return np.clip(pred, 1, 5) # Ensure predictions are within rating bounds

# Compute baseline predictions and convert into a dataframe
basepred_df = pd.DataFrame(
    index=train.index,
    columns=train.columns,
    dtype=float
)

for user in train.index:
    for item in train.columns:
        basepred_df.loc[user, item] = baseline_predictor(
            user, item,
            mean_rating,
            user_bias,
            item_bias
        )

basepred_df.round(2)

In [ ]:
# Add baseline prediction to test dataset
test['prediction'] = test.apply(
    lambda row: baseline_predictor(
        row['user'],
        row['item'],
        mean_rating,
        user_bias,
        item_bias
    ),
    axis=1
)

test.round(2)

In [ ]:
# Convert train dataset to long format
train_long = train.stack().reset_index()
train_long.columns = ['user', 'item', 'rating']

# Add baseline prediction to train_long dataset
train_long['prediction'] = train_long.apply(
    lambda row: baseline_predictor(
        row['user'],
        row['item'],
        mean_rating,
        user_bias,
        item_bias
    ),
    axis=1
)

train_long.round(2)

In [ ]:
# Calculate the RMSE for the baseline predictions of test dataset
test_bp_rmse = np.sqrt(
    mean_squared_error(
        test['rating'],
        test['prediction']
    )
)

test_bp_rmse.round(4)

In [ ]:
# Calculate the RMSE for the baseline predictions of train dataset
train_bp_rmse = np.sqrt(
    mean_squared_error(
        train_long['rating'],
        train_long['prediction']
    )
)

train_bp_rmse.round(4)

In [ ]:
# Compute percent improvement
test_pi = (1 - (test_bp_rmse / test_rmse)) * 100
train_pi = (1 - (train_bp_rmse / train_rmse)) * 100

test_pi.round(4), train_pi.round(4)


The baseline predictor achieved a lower RMSE than the global average predictor, indicating that accounting for user and item biases improved prediction accuracy.

In [ ]:
print(f"Global Average RMSE (Train): {train_rmse.round(4)}")
print(f"Baseline Predictor RMSE (Train): {train_bp_rmse.round(4)}")
print(f"Percent Improvement (Train): {train_pi.round(4)}\n")

print(f"Global Average RMSE (Test): {test_rmse.round(4)}")
print(f"Baseline Predictor RMSE (Test): {test_bp_rmse.round(4)}")
print(f"Percent Improvement (Test): {test_pi.round(4)}")

# Summary

This project explored the fundamental concept of global baseline predictors in recommendation systems using a toy user-oil paint ratings dataset. A user-item matrix containing missing ratings was constructed and then split into training and test datasets. The global average (mean) rating was first calculated and used as a simple prediction model. User and item biases were then computed to improve prediction accuracy by accounting for systemic differences in user preferences and item popularity.

To evaluate model performance, RMSE scores were calculated for both the global average predictor and bias adjusted baseline predictor. The results demonstrated that incorporating user and item biases improved prediction accuracy compared with using the global average alone. This simple yet effective approach provides a strong foundation for more advanced recommendation system algorithms.
